# PUSL 3205 - Large Language Models and Generative AI
## Lab: Sentiment Analysis Using Traditional Machine Learning

**Objective:** Practice basic NLP tasks  
**Topic:** Text Preprocessing, TF-IDF and Machine Learning  
**Libraries:** Pandas, NumPy, NLTK, Scikit-learn, Matplotlib, Seaborn  
**Dataset:** [UCI Sentiment Labelled Sentences](https://archive.ics.uci.edu/dataset/331/sentiment%2Blabelled%2Bsentences)

**Student Name:** _your name_  
**Student ID:** _your ID_

In [ ]:
import re
import io
import os
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

nltk.download('stopwords', quiet=True)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 100)

## Task 1: Download and load the dataset
The dataset has three files (Amazon, IMDB, Yelp). Each line is `sentence <TAB> label` where 1 = positive and 0 = negative. We combine them into one DataFrame.

In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip'
DATA_DIR = 'sentiment labelled sentences'

if not os.path.isdir(DATA_DIR):
    print('Downloading dataset...')
    data = urllib.request.urlopen(URL).read()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall('.')
    print('Done.')

files = {'amazon': 'amazon_cells_labelled.txt',
         'imdb': 'imdb_labelled.txt',
         'yelp': 'yelp_labelled.txt'}

frames = []
for source, fname in files.items():
    part = pd.read_csv(os.path.join(DATA_DIR, fname), sep='\t',
                       header=None, names=['text', 'label'])
    part['source'] = source
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.info())
print()
print('Missing values:\n', df.isnull().sum())
print()
print('Class distribution:\n', df['label'].value_counts())
sns.countplot(data=df, x='label')
plt.title('Class distribution (0 = negative, 1 = positive)')
plt.show()

## Task 2: Basic text preprocessing
Steps: lowercase -> remove unnecessary characters and punctuation -> remove extra whitespace -> store in a `clean_text` column.

In [ ]:
def preprocess(text):
    text = str(text).lower()                    # 1. lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)       # 2. remove punctuation, digits, symbols
    text = re.sub(r'\s+', ' ', text).strip()    # 3. remove extra whitespace
    return text

df['clean_text'] = df['text'].apply(preprocess)   # 4. clean-text column
df[['text', 'clean_text']].head()

**Compare original and preprocessed text**

In [ ]:
sample = df.sample(8, random_state=1)
for _, row in sample.iterrows():
    print('ORIGINAL :', row['text'])
    print('CLEANED  :', row['clean_text'])
    print('-' * 80)

df['orig_len'] = df['text'].str.len()
df['clean_len'] = df['clean_text'].str.len()
print('Average length - original:', round(df['orig_len'].mean(), 1),
      '| cleaned:', round(df['clean_len'].mean(), 1))

## Task 3: Explore the text data

In [ ]:
print('=== POSITIVE examples ===')
for t in df[df['label'] == 1]['text'].sample(5, random_state=42):
    print('+', t)

print('\n=== NEGATIVE examples ===')
for t in df[df['label'] == 0]['text'].sample(5, random_state=42):
    print('-', t)

### Frequently occurring words
We remove English stop words here so the chart shows meaningful words.

In [ ]:
stop_words = set(stopwords.words('english'))

def top_words(series, n=15):
    words = ' '.join(series).split()
    words = [w for w in words if w not in stop_words and len(w) > 1]
    return Counter(words).most_common(n)

pos_top = top_words(df[df['label'] == 1]['clean_text'])
neg_top = top_words(df[df['label'] == 0]['clean_text'])
all_top = top_words(df['clean_text'])

print('Overall   :', all_top)
print('Positive  :', pos_top)
print('Negative  :', neg_top)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in zip(axes, [pos_top, neg_top],
                                  ['Top words - Positive', 'Top words - Negative'],
                                  ['green', 'crimson']):
    words, counts = zip(*data)
    sns.barplot(x=list(counts), y=list(words), ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

### Differences between positive and negative reviews

In [ ]:
df['word_count'] = df['clean_text'].str.split().str.len()
print(df.groupby('label')['word_count'].describe())

sns.boxplot(data=df, x='label', y='word_count')
plt.title('Review length (words) by sentiment')
plt.show()

# Words most associated with each class (difference in frequency)
pos_c = Counter(w for w in ' '.join(df[df.label == 1]['clean_text']).split() if w not in stop_words)
neg_c = Counter(w for w in ' '.join(df[df.label == 0]['clean_text']).split() if w not in stop_words)
diff = {w: pos_c[w] - neg_c[w] for w in set(pos_c) | set(neg_c)}
ranked = sorted(diff.items(), key=lambda x: x[1])
print('\nMost negative-leaning words:', ranked[:10])
print('Most positive-leaning words:', ranked[-10:][::-1])

**Observations** *(check against your own output and edit if needed)*:
- Positive reviews tend to contain words such as *great, good, love, best, excellent, nice*.
- Negative reviews tend to contain words such as *bad, poor, disappointed, never, waste, worst* and negations like *not*.
- Both classes have a similar balance (about 500 positive and 500 negative per source), so accuracy is a fair metric.
- Review lengths are broadly similar, so sentiment is carried mostly by word choice rather than length.

## Task 4: Split the dataset
80% training / 20% testing, stratified so both sets keep the same class balance.

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))
print('Train class balance:\n', y_train.value_counts(normalize=True))

## Task 5: Create Bag-of-Words (BOW) representation
The vectorizer is fitted on the **training data only** to avoid data leakage.

In [ ]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print('Vocabulary size :', len(bow.vocabulary_))
print('Train matrix    :', X_train_bow.shape)
print('Test matrix     :', X_test_bow.shape)

# peek at the first document
print('\nExample sentence:', X_train.iloc[0])
nonzero = X_train_bow[0].nonzero()[1]
print('BOW counts      :', {bow.get_feature_names_out()[i]: int(X_train_bow[0, i]) for i in nonzero})

## Task 6: Train ML models
We train Logistic Regression and a linear SVM and compare them.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM (LinearSVC)': LinearSVC(),
}

for name, model in models.items():
    model.fit(X_train_bow, y_train)
    print(f'{name} trained.')

## Task 7: Evaluate the models
Metrics: Accuracy, Precision, Recall, F1-score.

In [ ]:
results = []
preds = {}
for name, model in models.items():
    y_pred = model.predict(X_test_bow)
    preds[name] = y_pred
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).set_index('Model').round(4)
results_df

In [ ]:
for name, y_pred in preds.items():
    print('=' * 60)
    print(name)
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

**Discussion:** Both models perform well above the 50% baseline using only word counts. The model with the higher F1-score is the better overall choice. Limitations of BOW: it ignores word order and context (e.g. *"not good"* vs *"good"*) and cannot understand sarcasm, which is where LLMs and transformer models improve on this approach.

## Task 8: Test the model with new sentences

In [ ]:
best_name = results_df['F1'].idxmax()
best_model = models[best_name]
print('Using best model:', best_name, '\n')

def predict_sentiment(sentence, model=best_model):
    cleaned = preprocess(sentence)
    vec = bow.transform([cleaned])
    label = model.predict(vec)[0]
    return 'Positive' if label == 1 else 'Negative'

new_sentences = [
    'I absolutely love this phone, the battery life is amazing!',
    'The food was cold and the service was terrible.',
    'This movie was a complete waste of time.',
    'Great quality and very fast delivery.',
    'I would never buy this product again.',
]

for s in new_sentences:
    print(f'{predict_sentiment(s):9s} | {s}')

In [ ]:
# Try your own sentence
my_sentence = 'The staff were friendly and the room was clean.'
print(predict_sentiment(my_sentence), '|', my_sentence)